# Cognix Phase 3: Region-Specific Pooling

**Purpose:** Test whether pooling brain vertices by region recovers cognitive dimensions that whole-brain mean pooling misses — especially emotional arousal (Brain−LLaMA = −0.070 in Round 2).

**No GPU needed.** Uses cached pooled vectors (20,484-d) and pre-computed similarities from Round 2.

**Key questions:**
1. Does limbic-only similarity recover emotional arousal?
2. Does prefrontal-only similarity best separate cognitive load?
3. Does motor-only similarity best capture sensorimotor?
4. Which regions diverge most from LLaMA?
5. Should the distilled model output a structured embedding (per-region scores) or a black box vector?

In [ ]:
# ---- Setup ----
!pip install -q nilearn nibabel

# Clone repo (or pull latest if already cloned)
!git clone https://github.com/gdavid7/cognix.git /content/cognix 2>/dev/null || (cd /content/cognix && git pull)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import cosine as cosine_dist
from pathlib import Path

PAIRS_PATH = '/content/cognix/data/validation_pairs_r2.jsonl'
RESULTS_PATH = '/content/cognix/data/r2_results.jsonl'
POOLED_DIR = Path('/content/drive/MyDrive/cognix_cache/pooled_vectors')

def text_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()[:16]

# Verify paths exist
assert Path(PAIRS_PATH).exists(), f'Missing: {PAIRS_PATH}'
assert Path(RESULTS_PATH).exists(), f'Missing: {RESULTS_PATH}'
assert POOLED_DIR.exists(), f'Missing: {POOLED_DIR}'
print('All paths verified.')

## 1. Load Destrieux atlas for fsaverage5

Map each of the 20,484 cortical vertices to a brain region. The Destrieux atlas has 74 regions per hemisphere.

In [ ]:
import nibabel as nib
from nilearn import datasets

# Fetch Destrieux atlas (downloads fsaverage-resolution .annot files)
atlas = datasets.fetch_atlas_surf_destrieux()

# Read annotation files with nibabel
lh_labels_full, _, lh_names = nib.freesurfer.read_annot(atlas['map_left'])
rh_labels_full, _, rh_names = nib.freesurfer.read_annot(atlas['map_right'])

# Decode byte strings to regular strings
lh_names = [n.decode() if isinstance(n, bytes) else n for n in lh_names]
rh_names = [n.decode() if isinstance(n, bytes) else n for n in rh_names]

print(f'Full atlas (fsaverage): {len(lh_labels_full)} LH + {len(rh_labels_full)} RH vertices')
print(f'Region labels: {len(lh_names)}')

# Downsample to fsaverage5: FreeSurfer ico meshes preserve lower-order
# vertices at the start, so the first 10242 vertices of fsaverage (ico7)
# correspond exactly to fsaverage5 (ico5) vertices.
N_FS5 = 10242
lh_labels = lh_labels_full[:N_FS5]
rh_labels = rh_labels_full[:N_FS5]
combined_labels = np.concatenate([lh_labels, rh_labels])

assert len(combined_labels) == 20484, f'Expected 20484, got {len(combined_labels)}'

# Both hemispheres use the same region name list
region_names_list = lh_names

print(f'Downsampled to fsaverage5: {len(combined_labels)} vertices')
print(f'Unique labels: {len(np.unique(combined_labels))}')
print(f'\nAll Destrieux region names:')
for i, name in enumerate(region_names_list):
    count = np.sum(combined_labels == i)
    if count > 0:
        print(f'  [{i:>2}] {name:<45} {count:>5} vertices')

## 2. Define cognitive region groups

Group Destrieux regions into 6 cognitive categories based on neuroscience literature.

In [ ]:
# Map each Destrieux region name to a cognitive group.
# Regions not listed go to 'other' (medial wall, background).
#
# Groupings based on functional neuroanatomy:
#   prefrontal: executive function, cognitive control, working memory
#   limbic: emotion processing, memory encoding, interoception
#   motor: motor planning and execution, somatosensory processing
#   parietal: spatial attention, visuospatial processing, sensory integration
#   temporal: language comprehension, auditory processing, social cognition
#   visual: visual processing, scene recognition, object recognition

REGION_NAME_TO_GROUP = {
    # ---- Prefrontal ----
    'G_and_S_frontomargin': 'prefrontal',
    'G_and_S_transv_frontopol': 'prefrontal',
    'G_front_inf-Opercular': 'prefrontal',
    'G_front_inf-Orbital': 'prefrontal',
    'G_front_inf-Triangul': 'prefrontal',
    'G_front_middle': 'prefrontal',
    'G_front_sup': 'prefrontal',
    'G_orbital': 'prefrontal',
    'G_rectus': 'prefrontal',
    'G_subcallosal': 'prefrontal',
    'S_front_inf': 'prefrontal',
    'S_front_middle': 'prefrontal',
    'S_front_sup': 'prefrontal',
    'S_orbital_lateral': 'prefrontal',
    'S_orbital_med-olfact': 'prefrontal',
    'S_orbital-H_Shaped': 'prefrontal',
    # ---- Limbic ----
    'G_and_S_cingul-Ant': 'limbic',
    'G_and_S_cingul-Mid-Ant': 'limbic',
    'G_and_S_cingul-Mid-Post': 'limbic',
    'G_cingul-Post-dorsal': 'limbic',
    'G_cingul-Post-ventral': 'limbic',
    'G_oc-temp_med-Parahip': 'limbic',
    'G_Ins_lg_and_S_cent_ins': 'limbic',
    'G_insular_short': 'limbic',
    'S_cingul-Marginalis': 'limbic',
    'S_circular_insula_ant': 'limbic',
    'S_circular_insula_inf': 'limbic',
    'S_circular_insula_sup': 'limbic',
    'S_pericallosal': 'limbic',
    'S_suborbital': 'limbic',
    # ---- Motor/Somatosensory ----
    'G_and_S_paracentral': 'motor',
    'G_and_S_subcentral': 'motor',
    'G_precentral': 'motor',
    'G_postcentral': 'motor',
    'S_central': 'motor',
    'S_precentral-inf-part': 'motor',
    'S_precentral-sup-part': 'motor',
    'S_postcentral': 'motor',
    # ---- Parietal ----
    'G_pariet_inf-Angular': 'parietal',
    'G_pariet_inf-Supramar': 'parietal',
    'G_parietal_sup': 'parietal',
    'G_precuneus': 'parietal',
    'S_intrapariet_and_P_trans': 'parietal',
    'S_subparietal': 'parietal',
    'S_parieto_occipital': 'parietal',
    'S_interm_prim-Jensen': 'parietal',
    # ---- Temporal ----
    'G_temp_sup-G_T_transv': 'temporal',
    'G_temp_sup-Lateral': 'temporal',
    'G_temp_sup-Plan_polar': 'temporal',
    'G_temp_sup-Plan_tempo': 'temporal',
    'G_temporal_inf': 'temporal',
    'G_temporal_middle': 'temporal',
    'Lat_Fis-ant-Horizont': 'temporal',
    'Lat_Fis-ant-Vertical': 'temporal',
    'Lat_Fis-post': 'temporal',
    'Pole_temporal': 'temporal',
    'S_temporal_inf': 'temporal',
    'S_temporal_sup': 'temporal',
    'S_temporal_transverse': 'temporal',
    'S_collat_transv_ant': 'temporal',
    # ---- Visual ----
    'G_and_S_occipital_inf': 'visual',
    'G_cuneus': 'visual',
    'G_oc-temp_lat-fusifor': 'visual',
    'G_oc-temp_med-Lingual': 'visual',
    'G_occipital_middle': 'visual',
    'G_occipital_sup': 'visual',
    'Pole_occipital': 'visual',
    'S_calcarine': 'visual',
    'S_oc_middle_and_Lunatus': 'visual',
    'S_oc_sup_and_transversal': 'visual',
    'S_oc-temp_lat': 'visual',
    'S_oc-temp_med_and_Lingual': 'visual',
    'S_collat_transv_post': 'visual',
}

GROUP_NAMES = ['prefrontal', 'limbic', 'motor', 'parietal', 'temporal', 'visual']

# Build vertex index -> cognitive group mapping
# For each vertex, look up its Destrieux label, then map to cognitive group
vertex_to_group = []
unmatched = set()
for label_id in combined_labels:
    if label_id < len(region_names_list):
        name = region_names_list[label_id]
    else:
        name = 'unknown'
    group = REGION_NAME_TO_GROUP.get(name, 'other')
    vertex_to_group.append(group)
    if group == 'other' and name not in ('Background+FreeSurfer_Defined_Medial_Wall', 'Medial_wall', 'Unknown', 'unknown'):
        unmatched.add(name)

vertex_to_group = np.array(vertex_to_group)

# Build group -> vertex indices
group_indices = {}
for group in GROUP_NAMES + ['other']:
    group_indices[group] = np.where(vertex_to_group == group)[0]

# Report
print('COGNITIVE REGION GROUPS')
print(f'{"Group":<15} {"Vertices":>8}  {"% of brain":>10}')
print('-' * 38)
total_assigned = 0
for group in GROUP_NAMES:
    n = len(group_indices[group])
    total_assigned += n
    print(f'{group:<15} {n:>8}  {100*n/20484:>9.1f}%')
n_other = len(group_indices['other'])
print(f'{"other":<15} {n_other:>8}  {100*n_other/20484:>9.1f}%')
print(f'{"TOTAL":<15} {total_assigned + n_other:>8}')

if unmatched:
    print(f'\nWARNING: {len(unmatched)} regions not assigned to any cognitive group:')
    for name in sorted(unmatched):
        print(f'  {name}')
else:
    print(f'\nAll non-background regions assigned to cognitive groups.')

## 3. Load data

Load pairs, pre-computed similarities from Round 2, and cached brain vectors.

In [ ]:
# Load pairs (need text content for hashing)
pairs = []
with open(PAIRS_PATH) as f:
    for line in f:
        pairs.append(json.loads(line))

# Load pre-computed similarities from Round 2
r2_results = []
with open(RESULTS_PATH) as f:
    for line in f:
        r2_results.append(json.loads(line))

assert len(pairs) == len(r2_results), f'Mismatch: {len(pairs)} pairs vs {len(r2_results)} results'

# Extract pre-computed similarities
semantic_sims = np.array([r['sim_semantic'] for r in r2_results])
brain_sims = np.array([r['sim_brain'] if r['sim_brain'] is not None else np.nan for r in r2_results])
llama_sims = np.array([r['sim_llama'] if r['sim_llama'] is not None else np.nan for r in r2_results])

print(f'Loaded {len(pairs)} pairs with pre-computed similarities')
print(f'  Semantic: {np.sum(~np.isnan(semantic_sims))}/{ len(pairs)} valid')
print(f'  Brain:    {np.sum(~np.isnan(brain_sims))}/{len(pairs)} valid')
print(f'  LLaMA:    {np.sum(~np.isnan(llama_sims))}/{len(pairs)} valid')

# Load brain vectors (20484-d pooled vectors)
brain_cache = {}
for npy_file in POOLED_DIR.glob('*.npy'):
    brain_cache[npy_file.stem] = np.load(npy_file)
print(f'\nLoaded {len(brain_cache)} brain vectors from Drive')

# Verify a sample vector has the right shape
sample = next(iter(brain_cache.values()))
assert sample.shape == (20484,), f'Expected (20484,), got {sample.shape}'
print(f'Vector shape verified: {sample.shape}')

## 4. Compute per-region similarity for all pairs

For each pair and each cognitive region, extract the vertices belonging to that region from both brain vectors and compute cosine similarity on the sub-vectors.

In [ ]:
region_sims = {r: [] for r in GROUP_NAMES}

for p in pairs:
    ha = text_hash(p['text_a'])
    hb = text_hash(p['text_b'])
    va = brain_cache.get(ha)
    vb = brain_cache.get(hb)

    for region in GROUP_NAMES:
        idx = group_indices[region]
        if va is not None and vb is not None and len(idx) > 1:
            sub_a = va[idx]
            sub_b = vb[idx]
            # Guard against zero vectors (would cause division by zero in cosine)
            if np.linalg.norm(sub_a) > 0 and np.linalg.norm(sub_b) > 0:
                region_sims[region].append(1.0 - cosine_dist(sub_a, sub_b))
            else:
                region_sims[region].append(np.nan)
        else:
            region_sims[region].append(np.nan)

for r in GROUP_NAMES:
    region_sims[r] = np.array(region_sims[r])

# Summary
print('Mean similarity per region (all pairs):')
for r in GROUP_NAMES:
    valid = np.sum(~np.isnan(region_sims[r]))
    print(f'  {r:<15} mean={np.nanmean(region_sims[r]):.3f}  std={np.nanstd(region_sims[r]):.3f}  (n={valid})')
print(f'  {"whole_brain":<15} mean={np.nanmean(brain_sims):.3f}  std={np.nanstd(brain_sims):.3f}')
print(f'  {"llama":<15} mean={np.nanmean(llama_sims):.3f}  std={np.nanstd(llama_sims):.3f}')
print(f'  {"semantic":<15} mean={np.nanmean(semantic_sims):.3f}  std={np.nanstd(semantic_sims):.3f}')

## 5. Which region best captures each cognitive category?

For each handcrafted divergence category, compute mean similarity per region. If region pooling works:
- Limbic should be highest for emotional arousal
- Prefrontal should be highest for cognitive load
- Motor should be highest for sensorimotor
- Parietal/visual should be highest for spatial scene

In [ ]:
divergence_cats = ['cognitive_load', 'emotional_arousal', 'sensorimotor',
                   'spatial_scene', 'syntactic_complexity', 'narrative_suspense',
                   'theory_of_mind']
all_cats = divergence_cats + ['control_same_topic_diff_processing', 'paraphrase', 'unrelated']

print('MEAN SIMILARITY PER REGION PER CATEGORY')
header = f'{"Category":<28}' + ''.join(f'{r:>11}' for r in GROUP_NAMES) + f'{"whole":>11}{"llama":>11}{"sem":>11}'
print(header)
print('-' * len(header))

for cat in all_cats:
    mask = np.array([p['category'] == cat and p['source'] == 'handcrafted' for p in pairs])
    if mask.sum() < 3:
        continue
    row = f'{cat:<28}'
    for r in GROUP_NAMES:
        row += f'{np.nanmean(region_sims[r][mask]):>11.3f}'
    row += f'{np.nanmean(brain_sims[mask]):>11.3f}'
    row += f'{np.nanmean(llama_sims[mask]):>11.3f}'
    row += f'{np.nanmean(semantic_sims[mask]):>11.3f}'
    print(row)

In [ ]:
# Which region has the HIGHEST similarity for each divergence category?
# And which region diverges most from LLaMA?

print('BEST REGION PER CATEGORY')
print(f'{"Category":<25} {"Best region":>12} {"Rgn sim":>8} {"Whole":>8} {"LLaMA":>8} {"Rgn-LLaMA":>10}')
print('-' * 78)

for cat in divergence_cats:
    mask = np.array([p['category'] == cat and p['source'] == 'handcrafted' for p in pairs])
    if mask.sum() < 3:
        continue

    best_region = max(GROUP_NAMES, key=lambda r: np.nanmean(region_sims[r][mask]))
    best_sim = np.nanmean(region_sims[best_region][mask])
    whole = np.nanmean(brain_sims[mask])
    llama = np.nanmean(llama_sims[mask])

    print(f'{cat:<25} {best_region:>12} {best_sim:>8.3f} {whole:>8.3f} {llama:>8.3f} {best_sim - llama:>+10.3f}')

## 6. Does limbic pooling recover emotional arousal?

Emotional arousal was the only category where whole-brain brain sim < LLaMA sim (−0.070). If limbic-only pooling shows brain sim > LLaMA sim, region pooling recovers the signal.

In [ ]:
emo_mask = np.array([p['category'] == 'emotional_arousal' and p['source'] == 'handcrafted' for p in pairs])

print('EMOTIONAL AROUSAL — REGION COMPARISON')
print(f'  N pairs: {emo_mask.sum()}')
print()

emo_llama = np.nanmean(llama_sims[emo_mask])
emo_whole = np.nanmean(brain_sims[emo_mask])

print(f'  {"Metric":<15} {"Sim":>8} {"vs LLaMA":>10}')
print(f'  {"-"*35}')
print(f'  {"semantic":<15} {np.nanmean(semantic_sims[emo_mask]):>8.3f}')
print(f'  {"llama":<15} {emo_llama:>8.3f}  {"(baseline)":>10}')
print(f'  {"whole_brain":<15} {emo_whole:>8.3f}  {emo_whole - emo_llama:>+10.3f}')
print()
for r in GROUP_NAMES:
    rsim = np.nanmean(region_sims[r][emo_mask])
    marker = '  <<<' if r == 'limbic' else ''
    print(f'  {r:<15} {rsim:>8.3f}  {rsim - emo_llama:>+10.3f}{marker}')

print()
limbic_emo = np.nanmean(region_sims['limbic'][emo_mask])
if limbic_emo > emo_llama and emo_whole < emo_llama:
    print('RESULT: Limbic pooling RECOVERS emotional arousal that whole-brain missed.')
elif limbic_emo > emo_whole:
    print('RESULT: Limbic pooling improves over whole-brain, but signal is partial.')
else:
    print('RESULT: Limbic pooling does NOT improve emotional arousal.')

## 7. Region−LLaMA divergence per category

For each region, compute Brain−LLaMA per category. Positive = the brain region sees more similarity than LLaMA.

In [ ]:
print('REGION−LLaMA DIVERGENCE PER CATEGORY')
print('(Positive = region sees MORE similarity than LLaMA)')
header = f'{"Category":<28}' + ''.join(f'{r:>11}' for r in GROUP_NAMES) + f'{"whole":>11}'
print(header)
print('-' * len(header))

for cat in all_cats:
    mask = np.array([p['category'] == cat and p['source'] == 'handcrafted' for p in pairs])
    if mask.sum() < 3:
        continue
    llama_mean = np.nanmean(llama_sims[mask])
    row = f'{cat:<28}'
    for r in GROUP_NAMES:
        diff = np.nanmean(region_sims[r][mask]) - llama_mean
        row += f'{diff:>+11.3f}'
    whole_diff = np.nanmean(brain_sims[mask]) - llama_mean
    row += f'{whole_diff:>+11.3f}'
    print(row)

## 8. Region correlation with semantic similarity

Which region is most independent from semantic similarity? Lower r = captures something more different.

In [ ]:
valid = ~np.isnan(brain_sims)

print('PEARSON r: REGION vs SEMANTIC SIMILARITY')
print(f'{"Region":<15} {"Pearson r":>10} {"p-value":>12}')
print('-' * 40)

for r in GROUP_NAMES:
    rvalid = valid & ~np.isnan(region_sims[r])
    if rvalid.sum() > 10:
        rp, pp = stats.pearsonr(semantic_sims[rvalid], region_sims[r][rvalid])
        print(f'{r:<15} {rp:>10.4f} {pp:>12.2e}')

rp, pp = stats.pearsonr(semantic_sims[valid], brain_sims[valid])
print(f'{"whole_brain":<15} {rp:>10.4f} {pp:>12.2e}')
rp, pp = stats.pearsonr(semantic_sims[valid], llama_sims[valid])
print(f'{"llama":<15} {rp:>10.4f} {pp:>12.2e}')

## 9. Heatmap: Region−LLaMA divergence by category

In [ ]:
regions_to_plot = GROUP_NAMES + ['whole_brain']

matrix = np.zeros((len(all_cats), len(regions_to_plot)))
valid_cats = []
for i, cat in enumerate(all_cats):
    mask = np.array([p['category'] == cat and p['source'] == 'handcrafted' for p in pairs])
    if mask.sum() < 3:
        continue
    valid_cats.append(cat)
    llama_mean = np.nanmean(llama_sims[mask])
    for j, r in enumerate(regions_to_plot):
        if r == 'whole_brain':
            matrix[len(valid_cats)-1, j] = np.nanmean(brain_sims[mask]) - llama_mean
        else:
            matrix[len(valid_cats)-1, j] = np.nanmean(region_sims[r][mask]) - llama_mean

matrix = matrix[:len(valid_cats), :]

fig, ax = plt.subplots(figsize=(12, 8))
vmax = max(abs(matrix.min()), abs(matrix.max()), 0.1)
im = ax.imshow(matrix, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(regions_to_plot)))
ax.set_xticklabels(regions_to_plot, rotation=45, ha='right')
ax.set_yticks(range(len(valid_cats)))
ax.set_yticklabels(valid_cats)
ax.set_title('Region−LLaMA divergence per category\n(Red = brain region > LLaMA, Blue = brain region < LLaMA)')
plt.colorbar(im, label='Brain region sim − LLaMA sim')

for i in range(len(valid_cats)):
    for j in range(len(regions_to_plot)):
        ax.text(j, i, f'{matrix[i,j]:+.3f}', ha='center', va='center', fontsize=7,
                color='white' if abs(matrix[i,j]) > vmax * 0.6 else 'black')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/cognix_cache/region_heatmap.png', dpi=150)
plt.show()
print('Saved to Drive: region_heatmap.png')

## 10. Summary and decision

In [ ]:
print('PHASE 3 SUMMARY')
print('=' * 60)

# Check if limbic recovered emotional arousal
emo_mask = np.array([p['category'] == 'emotional_arousal' and p['source'] == 'handcrafted' for p in pairs])
emo_llama = np.nanmean(llama_sims[emo_mask])
limbic_recovered = np.nanmean(region_sims['limbic'][emo_mask]) > emo_llama

# Check if regions show expected category-specific patterns
expected_best = {
    'cognitive_load': 'prefrontal',
    'emotional_arousal': 'limbic',
    'sensorimotor': 'motor',
    'spatial_scene': 'parietal',
}
matches = 0
for cat, expected_region in expected_best.items():
    mask = np.array([p['category'] == cat and p['source'] == 'handcrafted' for p in pairs])
    if mask.sum() < 3:
        continue
    actual_best = max(GROUP_NAMES, key=lambda r: np.nanmean(region_sims[r][mask]))
    match = actual_best == expected_region
    matches += int(match)
    status = 'MATCH' if match else f'MISMATCH (got {actual_best})'
    print(f'  {cat}: expected {expected_region}, {status}')

print(f'\n  Region-category matches: {matches}/{len(expected_best)}')
print(f'  Limbic recovered emotional arousal: {limbic_recovered}')

print()
if matches >= 3 and limbic_recovered:
    print('DECISION: Region pooling shows clear category-specific patterns.')
    print('  -> Distilled model should output STRUCTURED embedding with per-region scores.')
elif matches >= 2 or limbic_recovered:
    print('DECISION: Region pooling shows partial patterns.')
    print('  -> Consider structured embedding but include whole-brain as fallback.')
else:
    print('DECISION: Region pooling does not show clear category-specific patterns.')
    print('  -> Distilled model should output BLACK BOX embedding (single vector).')
    print('  -> The brain signal is diffuse, not cleanly region-specific.')

In [ ]:
# ---- Save results ----
results = []
for i, p in enumerate(pairs):
    entry = {
        'id': p['id'],
        'source': p['source'],
        'category': p['category'],
        'sim_semantic': float(semantic_sims[i]),
        'sim_llama': float(llama_sims[i]) if not np.isnan(llama_sims[i]) else None,
        'sim_whole_brain': float(brain_sims[i]) if not np.isnan(brain_sims[i]) else None,
    }
    for r in GROUP_NAMES:
        val = region_sims[r][i]
        entry[f'sim_{r}'] = float(val) if not np.isnan(val) else None
    results.append(entry)

out_path = '/content/drive/MyDrive/cognix_cache/r3_region_results.jsonl'
with open(out_path, 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')
print(f'Saved {len(results)} results to {out_path}')